# Held-Out Test: `TodPennnyOIThresholdStrategy`

This notebook runs the final held-out workflow only:

1. Fit `TodPennnyOIThresholdStrategy` on all of `data-train`
2. Evaluate the frozen fitted strategy on all of `data-test`

It does not use the internal `train_frac` split from the development pipeline.

## Colab Setup

This cell prepares a clean Colab environment. It clones the repo into `/content/optimal-execution`, mounts Google Drive, copies the held-out `train` and `test` CSV folders into `data-train` and `data-test`, and installs the Python dependencies needed to run the notebook.

If you are running locally and already have the repo plus data directories set up, skip this cell.

In [6]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print('Local environment detected. Skip this cell if your repo and data are already set up.')
else:
    drive.mount('/content/drive')
    os.chdir('/content')

    repo_dir = Path('/content/optimal-execution')
    if repo_dir.exists():
        shutil.rmtree(repo_dir)

    subprocess.run(
        ['git', 'clone', 'https://github.com/MatteoPerona/optimal-execution.git'],
        check=True,
    )
    os.chdir(repo_dir)

    shutil.rmtree('data-train', ignore_errors=True)
    shutil.rmtree('data-test', ignore_errors=True)
    shutil.copytree('/content/drive/MyDrive/cs5259data/train', 'data-train')
    shutil.copytree('/content/drive/MyDrive/cs5259data/test', 'data-test')

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'scipy', 'matplotlib', 'jupyter', 'ipython'],
        check=True,
    )

    print(f'Working directory: {Path.cwd()}')
    print('Train files:', sorted(os.listdir('data-train'))[:5])
    print('Test files:', sorted(os.listdir('data-test'))[:5])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/optimal-execution
Train files: ['AAPL_5levels_train.csv', 'AMZN_5levels_train.csv', 'GOOG_5levels_train.csv', 'INTC_5levels_train.csv', 'MSFT_5levels_train.csv']
Test files: ['AAPL_5levels_test.csv', 'AMZN_5levels_test.csv', 'GOOG_5levels_test.csv', 'INTC_5levels_test.csv', 'MSFT_5levels_test.csv']


In [7]:
import os
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == 'heldout_test':
    repo_root = repo_root.parent
elif repo_root.name != 'optimal-execution' and (repo_root / 'optimal-execution').exists():
    repo_root = repo_root / 'optimal-execution'

os.chdir(repo_root)
print(f'Working directory: {Path.cwd()}')

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11})

from utils.config import DEFAULT_CONFIG
from utils.strategy import TodPennnyOIThresholdStrategy
from utils.evaluation import print_results, plot_results
from heldout_test import run_external_test_experiment

Working directory: /content/optimal-execution


ModuleNotFoundError: No module named 'heldout_test'

In [ ]:
heldout_config = DEFAULT_CONFIG.copy()
heldout_config['stocks'] = ['AAPL', 'AMZN', 'GOOG', 'INTC', 'MSFT']
heldout_config['train_data_dir'] = 'data-train'
heldout_config['test_data_dir'] = 'data-test'

heldout_config

In [ ]:
todpennny_heldout_results = run_external_test_experiment(
    TodPennnyOIThresholdStrategy,
    heldout_config,
    signal_fn='oi',
)

todpennny_heldout_results['fitted']

In [ ]:
print_results(
    todpennny_heldout_results['test_all'],
    strategy_name='todpennny',
)
plot_results(
    todpennny_heldout_results['test_all'],
    strategy_name='todpennny',
)